# Midsem Question 5 — Kneser-Ney Smoothing

**Sentences (no start/end markers):**
- S1: `I like NLP`
- S2: `I like AI`
- S3: `You like NLP`
- S4: `You love AI`
- S5: `We love NLP`

**Discount:** `d = 0.5`

**Task:** Compute `P_KN(AI | love)` and `P_KN(NLP | like)`.

Kneser-Ney bigram:

P_KN(wᵢ | wᵢ₋₁) = max(c(wᵢ₋₁, wᵢ) − d, 0) / c(wᵢ₋₁)  +  λ(wᵢ₋₁) × P_KN(wᵢ)

where
- λ(wᵢ₋₁) = d × (number of distinct words following wᵢ₋₁) / c(wᵢ₋₁)
- P_KN(w) = (number of distinct left contexts of w) / (total number of distinct bigram types)

In [1]:
from collections import Counter, defaultdict

sentences = [
    "I like NLP",
    "I like AI",
    "You like NLP",
    "You love AI",
    "We love NLP",
]
d = 0.5

                                   
bigram_counts = Counter()
unigram_counts = Counter()
for s in sentences:
    words = s.split()
    for w in words:
        unigram_counts[w] += 1
    for i in range(len(words) - 1):
        bigram_counts[(words[i], words[i + 1])] += 1

distinct_bigram_types = len(bigram_counts)
print("Bigram counts:")
for bg, c in sorted(bigram_counts.items()):
    print(f"  {bg} : {c}")
print(f"\nTotal distinct bigram types = {distinct_bigram_types}")
print("\nUnigram counts:", dict(sorted(unigram_counts.items())))

                                 
followers = defaultdict(set)
for (a, b) in bigram_counts:
    followers[a].add(b)
                                     
left_contexts = defaultdict(set)
for (a, b) in bigram_counts:
    left_contexts[b].add(a)

print("\nDistinct followers per word:", {k: sorted(v) for k, v in sorted(followers.items())})
print("Distinct left contexts per word:", {k: sorted(v) for k, v in sorted(left_contexts.items())})

Bigram counts:
  ('I', 'like') : 2
  ('We', 'love') : 1
  ('You', 'like') : 1
  ('You', 'love') : 1
  ('like', 'AI') : 1
  ('like', 'NLP') : 2
  ('love', 'AI') : 1
  ('love', 'NLP') : 1

Total distinct bigram types = 8

Unigram counts: {'AI': 2, 'I': 2, 'NLP': 3, 'We': 1, 'You': 2, 'like': 3, 'love': 2}

Distinct followers per word: {'I': ['like'], 'We': ['love'], 'You': ['like', 'love'], 'like': ['AI', 'NLP'], 'love': ['AI', 'NLP']}
Distinct left contexts per word: {'AI': ['like', 'love'], 'NLP': ['like', 'love'], 'like': ['I', 'You'], 'love': ['We', 'You']}


In [2]:
                                        
def p_kn_unigram(w):
    return len(left_contexts.get(w, set())) / distinct_bigram_types


                                       
def p_kn_bigram(w_prev, w_cur):
    c_bigram = bigram_counts.get((w_prev, w_cur), 0)
    c_prev = unigram_counts.get(w_prev, 0)
    if c_prev == 0:
        return 0.0
    first = max(c_bigram - d, 0) / c_prev
    lam = d * len(followers.get(w_prev, set())) / c_prev
    return first + lam * p_kn_unigram(w_cur)


                               
q1 = p_kn_bigram("love", "AI")
q2 = p_kn_bigram("like", "NLP")

print("=" * 55)
print("P_KN(AI | love):")
print(f"  c(love, AI) = {bigram_counts[('love','AI')]}, c(love) = {unigram_counts['love']}")
print(f"  first term  = max({bigram_counts[('love','AI')]} - {d}, 0) / {unigram_counts['love']} = {max(bigram_counts[('love','AI')]-d,0)/unigram_counts['love']:.6f}")
print(f"  lambda(love)= {d} x {len(followers['love'])} / {unigram_counts['love']} = {d*len(followers['love'])/unigram_counts['love']:.6f}")
print(f"  P_KN(AI)    = {len(left_contexts['AI'])}/{distinct_bigram_types} = {p_kn_unigram('AI'):.6f}")
print(f"  => P_KN(AI | love) = {q1:.6f}")
print()
print("P_KN(NLP | like):")
print(f"  c(like, NLP) = {bigram_counts[('like','NLP')]}, c(like) = {unigram_counts['like']}")
print(f"  first term   = max({bigram_counts[('like','NLP')]} - {d}, 0) / {unigram_counts['like']} = {max(bigram_counts[('like','NLP')]-d,0)/unigram_counts['like']:.6f}")
print(f"  lambda(like) = {d} x {len(followers['like'])} / {unigram_counts['like']} = {d*len(followers['like'])/unigram_counts['like']:.6f}")
print(f"  P_KN(NLP)    = {len(left_contexts['NLP'])}/{distinct_bigram_types} = {p_kn_unigram('NLP'):.6f}")
print(f"  => P_KN(NLP | like) = {q2:.6f}")
print("=" * 55)

P_KN(AI | love):
  c(love, AI) = 1, c(love) = 2
  first term  = max(1 - 0.5, 0) / 2 = 0.250000
  lambda(love)= 0.5 x 2 / 2 = 0.500000
  P_KN(AI)    = 2/8 = 0.250000
  => P_KN(AI | love) = 0.375000

P_KN(NLP | like):
  c(like, NLP) = 2, c(like) = 3
  first term   = max(2 - 0.5, 0) / 3 = 0.500000
  lambda(like) = 0.5 x 2 / 3 = 0.333333
  P_KN(NLP)    = 2/8 = 0.250000
  => P_KN(NLP | like) = 0.583333


## Answer

With `d = 0.5`:

**P_KN(AI | love):**
- c(love, AI) = 1, c(love) = 2 → first term = max(1 − 0.5, 0)/2 = 0.25
- λ(love) = 0.5 × (distinct followers of love = {AI, NLP} = 2) / 2 = 0.5
- P_KN(AI) = (distinct left contexts of AI = {like, love} = 2) / 8 = 0.25
- P_KN(AI | love) = 0.25 + 0.5 × 0.25 = **0.375**

**P_KN(NLP | like):**
- c(like, NLP) = 2, c(like) = 3 → first term = max(2 − 0.5, 0)/3 = 0.5
- λ(like) = 0.5 × (distinct followers of like = {NLP, AI} = 2) / 3 ≈ 0.3333
- P_KN(NLP) = (distinct left contexts of NLP = {like, love} = 2) / 8 = 0.25
- P_KN(NLP | like) = 0.5 + (1/3) × 0.25 ≈ **0.5833**

**Results:** P_KN(AI | love) = 0.375 ;  P_KN(NLP | like) ≈ 0.5833.